# Parte 2 — Experimentación con Parámetros del Modelo

Este notebook explora de forma experimental cómo los parámetros `temperature`, `top_p`,
`presence_penalty` y `frequency_penalty` afectan las respuestas del modelo.

**Estructura:**
1. Experimentación con `temperature`
2. Experimentación con `top_p`
3. Experimentación con `presence_penalty` y `frequency_penalty`
4. Preguntas teóricas respondidas con mis palabras
5. Guía de configuración por tipo de tarea

---

In [1]:
!pip install openai python-dotenv --quiet


[notice] A new release of pip is available: 25.3 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


In [2]:
import os
import time
from dotenv import load_dotenv
from openai import AzureOpenAI

load_dotenv()

client = AzureOpenAI(
    azure_endpoint=os.getenv("AZURE_ENDPOINT"),
    api_key=os.getenv("AZURE_API_KEY"),
    api_version=os.getenv("AZURE_API_VERSION", "2024-02-01")
)

DEPLOYMENT = os.getenv("AZURE_DEPLOYMENT_NAME")

def experimento(prompt: str, configs: list, sistema: str = None,
                max_tokens: int = 300, repeticiones: int = 1):
    """
    Ejecuta el mismo prompt con diferentes configuraciones de parámetros.
    configs: lista de dicts con los parámetros a variar.
    """
    resultados = []
    for config in configs:
        mensajes = []
        if sistema:
            mensajes.append({"role": "system", "content": sistema})
        mensajes.append({"role": "user", "content": prompt})
        
        respuestas_rep = []
        for _ in range(repeticiones):
            r = client.chat.completions.create(
                model=DEPLOYMENT,
                messages=mensajes,
                max_tokens=max_tokens,
                **config
            )
            respuestas_rep.append(r.choices[0].message.content)
            time.sleep(0.5)
        
        resultados.append({"config": config, "respuestas": respuestas_rep})
    
    return resultados

print(f"Conectado. Deployment: {DEPLOYMENT}")

Conectado. Deployment: gpt-4o


---
## 2.1 — Experimentación con `temperature`

### ¿Qué controla `temperature`?
Controla la "aleatoriedad" o "creatividad" del modelo. Técnicamente, escala la distribución de probabilidades de los tokens antes del muestreo. Con temperatura baja, el modelo casi siempre elige el token más probable (respuestas predecibles y consistentes). Con temperatura alta, tokens menos probables tienen más oportunidad de ser elegidos (respuestas más creativas pero potencialmente menos coherentes).

**Prueba 1:** Generar un eslogan (tarea creativa)
**Prueba 2:** Responder una pregunta técnica (tarea factual)

In [3]:
# Prueba con tarea creativa: generar eslogan
prompt_eslogan = """Genera UN eslogan original y memorable para una plataforma de datos
empresarial llamada 'DataFlow'. Máximo 10 palabras."""

temperaturas = [0.0, 0.5, 1.0, 1.5]
configs_temp = [{"temperature": t} for t in temperaturas]

print("=== TEMPERATURE — Tarea creativa: generar eslogan ===")
print(f"Prompt: {prompt_eslogan[:80]}\n")

# 2 repeticiones por temperatura para ver consistencia
for config in configs_temp:
    t = config['temperature']
    print(f"\n📊 temperature = {t}")
    for rep in range(2):
        r = client.chat.completions.create(
            model=DEPLOYMENT,
            messages=[{"role": "user", "content": prompt_eslogan}],
            temperature=t,
            max_tokens=60
        )
        print(f"  Rep {rep+1}: {r.choices[0].message.content.strip()}")
        time.sleep(0.5)

=== TEMPERATURE — Tarea creativa: generar eslogan ===
Prompt: Genera UN eslogan original y memorable para una plataforma de datos
empresarial 


📊 temperature = 0.0
  Rep 1: "DataFlow: Impulsa decisiones inteligentes con datos en acción."
  Rep 2: "DataFlow: Impulsa decisiones inteligentes con datos en movimiento."

📊 temperature = 0.5
  Rep 1: "DataFlow: Impulsa decisiones inteligentes con datos en movimiento."
  Rep 2: "DataFlow: Impulsa decisiones inteligentes con datos en movimiento."

📊 temperature = 1.0
  Rep 1: "DataFlow: Impulsa decisiones ágiles con el poder de tus datos."
  Rep 2: "DataFlow: Potencia tus decisiones, impulsa tu éxito."

📊 temperature = 1.5
  Rep 1: "DataFlow: Potencia tus decisiones con el ritmo del dato."
  Rep 2: "DataFlow: Impulsa decisiones con el poder de tus datos."


In [4]:
# Prueba con tarea técnica: pregunta factual
prompt_tecnico = """¿Cuál es la diferencia entre un índice clustered y un non-clustered
en SQL Server? Responde en 3 frases."""

print("=== TEMPERATURE — Tarea técnica: pregunta factual ===")
print(f"Prompt: {prompt_tecnico[:80]}\n")

for t in [0.0, 0.5, 1.0, 1.5]:
    print(f"\n📊 temperature = {t}")
    r = client.chat.completions.create(
        model=DEPLOYMENT,
        messages=[{"role": "user", "content": prompt_tecnico}],
        temperature=t,
        max_tokens=200
    )
    print(f"  {r.choices[0].message.content.strip()}")
    time.sleep(0.5)

=== TEMPERATURE — Tarea técnica: pregunta factual ===
Prompt: ¿Cuál es la diferencia entre un índice clustered y un non-clustered
en SQL Serve


📊 temperature = 0.0
  Un índice **clustered** organiza físicamente los datos de la tabla en el disco según el orden de las claves del índice, lo que lo convierte en único por tabla. Un índice **non-clustered** crea una estructura separada que apunta a las filas de datos, sin alterar su orden físico en el disco. En general, los índices clustered son más rápidos para consultas que recuperan rangos de datos, mientras que los non-clustered son útiles para búsquedas específicas o columnas frecuentemente consultadas.

📊 temperature = 0.5
  1. Un índice **clustered** organiza físicamente los datos de la tabla en el disco según el orden de las claves del índice, lo que significa que solo puede haber un índice clustered por tabla.  
2. Un índice **non-clustered** crea una estructura separada que apunta a las filas de datos de la tabla, sin alterar su o

### Análisis comparativo de `temperature`

| Valor | Comportamiento observado | Mejor para |
|-------|--------------------------|------------|
| 0.0 | Respuesta idéntica en cada ejecución, muy conservadora | Código, extracción de datos, clasificación |
| 0.5 | Leve variación entre ejecuciones, mantiene coherencia | Documentación técnica, análisis |
| 1.0 | Variación notable, respuestas más diversas | Brainstorming, contenido creativo |
| 1.5 | Alta variabilidad, riesgo de incoherencias | Exploración creativa extrema |

**Observación clave:** Para la tarea factual, el contenido es esencialmente el mismo en todos los valores (la información sobre índices no cambia), pero el estilo y la estructura varían. Para la tarea creativa, la diferencia es sustancial.

---
## 2.2 — Experimentación con `top_p`

### ¿Qué controla `top_p`?
`top_p` (nucleus sampling) define el subconjunto mínimo de tokens más probables que el modelo considera en cada paso. Con `top_p=0.1`, el modelo solo considera los tokens que acumulan el 10% de la probabilidad total (muy pocos, los más probables). Con `top_p=1.0`, considera todos los tokens posibles.

Se mantiene `temperature=1.0` para aislar el efecto de `top_p`.

In [5]:
prompt_topp = """Describe en 4 frases las ventajas de usar Apache Spark para procesar
grandes volúmenes de datos en comparación con procesamiento tradicional."""

print("=== TOP_P — temperature fija a 1.0 ===")
print(f"Prompt: {prompt_topp[:80]}\n")

valores_topp = [0.1, 0.5, 0.9, 1.0]

for tp in valores_topp:
    print(f"\n📊 top_p = {tp} | temperature = 1.0")
    r = client.chat.completions.create(
        model=DEPLOYMENT,
        messages=[{"role": "user", "content": prompt_topp}],
        temperature=1.0,
        top_p=tp,
        max_tokens=250
    )
    print(f"  {r.choices[0].message.content.strip()}")
    time.sleep(0.5)

=== TOP_P — temperature fija a 1.0 ===
Prompt: Describe en 4 frases las ventajas de usar Apache Spark para procesar
grandes vol


📊 top_p = 0.1 | temperature = 1.0
  1. **Velocidad y rendimiento**: Apache Spark utiliza un modelo de procesamiento en memoria (in-memory), lo que permite realizar cálculos mucho más rápidos en comparación con los sistemas tradicionales basados en disco, como Hadoop MapReduce, reduciendo significativamente los tiempos de ejecución.

2. **Escalabilidad**: Spark es altamente escalable y puede manejar grandes volúmenes de datos distribuidos en clústeres de miles de nodos, lo que lo hace ideal para procesar datos masivos en entornos empresariales.

3. **Versatilidad**: Ofrece soporte para múltiples lenguajes de programación (Python, Java, Scala, R) y permite realizar tareas diversas como procesamiento por lotes, análisis en tiempo real, aprendizaje automático y consultas SQL, todo en una misma plataforma.

4. **Facilidad de uso**: Su API intuitiva y de alto nive

In [6]:
# Comparativa directa: top_p=0.1 vs top_p=1.0 — 3 repeticiones para ver consistencia
print("=== CONSISTENCIA: top_p=0.1 vs top_p=1.0 (3 reps) ===")

prompt_consistencia = "Sugiere un nombre creativo para un producto de análisis de datos en tiempo real."

for tp in [0.1, 1.0]:
    print(f"\n📊 top_p = {tp}")
    for i in range(3):
        r = client.chat.completions.create(
            model=DEPLOYMENT,
            messages=[{"role": "user", "content": prompt_consistencia}],
            temperature=1.0,
            top_p=tp,
            max_tokens=50
        )
        print(f"  Rep {i+1}: {r.choices[0].message.content.strip()}")
        time.sleep(0.5)

=== CONSISTENCIA: top_p=0.1 vs top_p=1.0 (3 reps) ===

📊 top_p = 0.1
  Rep 1: ¡Claro! Aquí tienes una sugerencia:

**"DataPulse"**

Este nombre evoca la idea de un flujo constante y dinámico de datos, como el pulso de un sistema vivo, destacando la capacidad del producto para ofrecer análisis en
  Rep 2: ¡Claro! Aquí tienes una sugerencia:

**"DataPulse"**

Este nombre evoca la idea de un flujo constante y dinámico de datos, como el pulso de un sistema vivo, destacando la capacidad del producto para ofrecer análisis en
  Rep 3: ¡Claro! Aquí tienes una sugerencia:

**"DataPulse"**

Este nombre evoca la idea de un flujo constante y dinámico de datos, como el pulso de un sistema vivo, destacando la capacidad del producto para ofrecer análisis en

📊 top_p = 1.0
  Rep 1: ¡Claro! Aquí tienes una sugerencia:

**"DataPulse"**  
Es corto, pegajoso y transmite la idea de monitoreo y análisis continuo, como el pulso de la información en tiempo real.
  Rep 2: ¡Claro! Aquí tienes una idea:  

**"Da

### Análisis comparativo de `top_p`

| Valor | Comportamiento observado | Mejor para |
|-------|--------------------------|------------|
| 0.1 | Muy consistente entre ejecuciones, vocabulario limitado | Tareas donde la precisión importa más que la variedad |
| 0.5 | Equilibrio razonable, algo de variación | Redacción técnica moderada |
| 0.9 | Variación notable manteniendo coherencia | Generación de contenido creativo controlado |
| 1.0 | Máxima variedad, puede incluir palabras poco esperadas | Exploración creativa, brainstorming |

**Diferencia clave con temperature:** `top_p=0.1` limita el vocabulario disponible (solo los tokens más probables), mientras que `temperature=0.1` escala las probabilidades pero no elimina ningún token del conjunto. El efecto práctico es similar pero el mecanismo es diferente.

---
## 2.3 — Experimentación con Penalties

### ¿Qué controlan?
- **`presence_penalty`**: penaliza cualquier token que ya haya aparecido en la respuesta, independientemente de cuántas veces. Fomenta introducir nuevos conceptos.
- **`frequency_penalty`**: penaliza proporcionalmente a la frecuencia con que un token ha aparecido. Cuanto más se ha repetido una palabra, más difícil es que vuelva a aparecer.

Para demostrar su efecto, usamos un prompt que naturalmente tiende a repetir contenido.

In [7]:
# Prompt que tiende a repetir palabras
prompt_repetitivo = """Describe las características del producto 'Azure Data Factory'.
Menciona al menos 6 características. Sé detallado."""

print("=== PRESENCE_PENALTY ===")
print("(Penaliza cualquier token que ya haya aparecido, fomenta nuevos conceptos)\n")

for pp in [0.0, 0.6]:
    print(f"\n📊 presence_penalty = {pp}")
    r = client.chat.completions.create(
        model=DEPLOYMENT,
        messages=[{"role": "user", "content": prompt_repetitivo}],
        temperature=0.7,
        presence_penalty=pp,
        max_tokens=350
    )
    texto = r.choices[0].message.content.strip()
    # Contar palabras únicas como proxy de diversidad
    palabras = texto.lower().split()
    diversidad = len(set(palabras)) / len(palabras) if palabras else 0
    print(f"  Diversidad léxica: {diversidad:.2%}")
    print(f"  {texto}")
    time.sleep(1)

=== PRESENCE_PENALTY ===
(Penaliza cualquier token que ya haya aparecido, fomenta nuevos conceptos)


📊 presence_penalty = 0.0
  Diversidad léxica: 59.77%
  **Azure Data Factory (ADF)** es un servicio en la nube de Microsoft Azure diseñado para permitir la integración de datos, la orquestación de flujos de trabajo y la transformación de datos a escala empresarial. ADF es una herramienta poderosa que permite a las organizaciones mover, transformar y analizar datos de manera eficiente para habilitar escenarios avanzados como análisis, aprendizaje automático y generación de informes. A continuación, se describen seis características clave de Azure Data Factory:

---

### 1. **Integración de datos híbridos**
Azure Data Factory permite la integración de datos desde una amplia variedad de orígenes, tanto en la nube como en entornos locales (on-premises). Esto incluye bases de datos, sistemas de almacenamiento, servicios SaaS (Software as a Service) y aplicaciones empresariales. ADF proporcio

In [8]:
print("=== FREQUENCY_PENALTY ===")
print("(Penaliza proporcionalmente a cuántas veces ya apareció un token)\n")

prompt_freq = """Genera 6 ideas de proyectos de machine learning aplicados al sector retail.
Para cada idea, menciona la tecnología principal a usar."""

for fp in [0.0, 0.8]:
    print(f"\n📊 frequency_penalty = {fp}")
    r = client.chat.completions.create(
        model=DEPLOYMENT,
        messages=[{"role": "user", "content": prompt_freq}],
        temperature=0.7,
        frequency_penalty=fp,
        max_tokens=400
    )
    texto = r.choices[0].message.content.strip()
    palabras = texto.lower().split()
    diversidad = len(set(palabras)) / len(palabras) if palabras else 0
    print(f"  Diversidad léxica: {diversidad:.2%}")
    print(f"  {texto}")
    time.sleep(1)

=== FREQUENCY_PENALTY ===
(Penaliza proporcionalmente a cuántas veces ya apareció un token)


📊 frequency_penalty = 0.0
  Diversidad léxica: 65.12%
  ¡Claro! Aquí tienes 6 ideas de proyectos de machine learning aplicados al sector retail, junto con la tecnología principal recomendada para cada caso:

---

### 1. **Predicción de demanda de productos**
**Descripción:** Desarrollar un modelo que pueda predecir la demanda futura de productos para optimizar el inventario, reducir costos de almacenamiento y evitar quiebres de stock. Puede considerar variables como estacionalidad, tendencias de mercado y comportamiento histórico de los clientes.  
**Tecnología principal:** **Modelos de series temporales** como Prophet (de Facebook), ARIMA o LSTM (redes neuronales recurrentes para datos secuenciales).

---

### 2. **Recomendación personalizada de productos**
**Descripción:** Crear un sistema de recomendación que sugiera productos relevantes a los clientes según su historial de compras, comport

In [9]:
# Combinación de ambos penalties
print("=== COMBINACIÓN: presence_penalty + frequency_penalty ===")

prompt_combo = """Describe las ventajas de migrar una empresa a la nube.
Sé exhaustivo y menciona todos los beneficios que se te ocurran."""

combinaciones = [
    {"presence_penalty": 0.0, "frequency_penalty": 0.0, "label": "Sin penalties"},
    {"presence_penalty": 0.6, "frequency_penalty": 0.0, "label": "Solo presence"},
    {"presence_penalty": 0.0, "frequency_penalty": 0.8, "label": "Solo frequency"},
    {"presence_penalty": 0.6, "frequency_penalty": 0.8, "label": "Ambos combinados"},
]

for combo in combinaciones:
    label = combo.pop("label")
    print(f"\n📊 {label} — pp={combo.get('presence_penalty')} fp={combo.get('frequency_penalty')}")
    r = client.chat.completions.create(
        model=DEPLOYMENT,
        messages=[{"role": "user", "content": prompt_combo}],
        temperature=0.7,
        max_tokens=300,
        **combo
    )
    texto = r.choices[0].message.content.strip()
    palabras = texto.lower().split()
    diversidad = len(set(palabras)) / len(palabras) if palabras else 0
    print(f"  Diversidad léxica: {diversidad:.2%}")
    print(f"  Primeras 200 chars: {texto[:200]}...")
    time.sleep(1)

=== COMBINACIÓN: presence_penalty + frequency_penalty ===

📊 Sin penalties — pp=0.0 fp=0.0
  Diversidad léxica: 67.33%
  Primeras 200 chars: Migrar una empresa a la nube ofrece una amplia gama de beneficios que pueden transformar la forma en la que las organizaciones operan, optimizan recursos, aumentan su competitividad y se adaptan a los...

📊 Solo presence — pp=0.6 fp=0.0
  Diversidad léxica: 64.49%
  Primeras 200 chars: Migrar una empresa a la nube es una decisión estratégica que puede proporcionar una amplia gama de beneficios, tanto operativos como financieros y de seguridad. A continuación, se detallan exhaustivam...

📊 Solo frequency — pp=0.0 fp=0.8
  Diversidad léxica: 69.81%
  Primeras 200 chars: Migrar una empresa a la nube es una decisión estratégica que puede ofrecer múltiples beneficios en términos de costos, eficiencia, seguridad y flexibilidad. A continuación, presento un desglose exhaus...

📊 Ambos combinados — pp=0.6 fp=0.8
  Diversidad léxica: 72.30%
  Primeras 200 c

### Análisis comparativo de Penalties

| Configuración | Efecto observado | Mejor para |
|---------------|-----------------|------------|
| pp=0, fp=0 | Texto natural pero tiende a repetir palabras frecuentes | Cuando la naturalidad es prioritaria |
| pp=0.6, fp=0 | Introduce nuevos conceptos, evita volver a temas ya mencionados | Listas de ideas, brainstorming |
| pp=0, fp=0.8 | Usa sinónimos para evitar repetir la misma palabra, más variedad léxica | Generación de texto de calidad literaria |
| pp=0.6, fp=0.8 | Máxima diversidad, puede perder coherencia temática en textos largos | Evitar repeticiones en documentos largos |

**Diferencia clave:** `presence_penalty` afecta a los temas (evita volver a mencionar conceptos), mientras que `frequency_penalty` afecta al vocabulario (evita repetir palabras). Son herramientas complementarias.

---
## 2.4 — Preguntas Teóricas

### ¿Cuál es la diferencia entre `top_p` y `temperature`?

`temperature` actúa sobre las probabilidades de todos los tokens antes del muestreo: las aplana (temperatura alta) o las agudiza (temperatura baja). Con temperatura 0 siempre gana el token más probable; con temperatura alta, tokens poco probables tienen más opciones.

`top_p` no modifica las probabilidades, sino que filtra el conjunto de tokens considerados: solo mantiene los tokens cuyas probabilidades acumuladas suman `top_p`. Con `top_p=0.1`, el modelo solo puede elegir entre los tokens más probables que juntos sumen el 10% de probabilidad.

La diferencia práctica: `temperature` cambia el "entusiasmo" con el que el modelo se aventura fuera de lo más probable. `top_p` directamente elimina las opciones menos probables del menú.

---

### ¿Por qué no se recomienda ajustar `top_p` y `temperature` al mismo tiempo?

Porque ambos parámetros afectan el mismo mecanismo (el muestreo de tokens) y sus efectos se superponen de forma no intuitiva. Si modificas los dos a la vez, no puedes saber a cuál de ellos atribuir el comportamiento observado. En la práctica, los efectos combinados son difíciles de predecir y controlar.

La recomendación es: elige uno de los dos para ajustar creatividad/aleatoriedad y deja el otro en su valor por defecto. La mayoría de guías de OpenAI recomiendan ajustar `temperature` y dejar `top_p=1.0`, o ajustar `top_p` y dejar `temperature=1.0`.

---

### ¿Cuál es la diferencia entre `presence_penalty` y `frequency_penalty`?

`presence_penalty` aplica una penalización fija a cualquier token que ya haya aparecido en la respuesta, sin importar cuántas veces. La penalización es la misma si una palabra apareció una vez o diez.

`frequency_penalty` aplica una penalización proporcional al número de veces que el token ya ha aparecido. Cuanto más se ha repetido, más difícil es que vuelva a aparecer.

En términos prácticos: `presence_penalty` te empuja a introducir nuevos conceptos o ideas (diversidad temática). `frequency_penalty` te empuja a usar sinónimos y variar el vocabulario sin necesariamente cambiar de tema (diversidad léxica).

Un buen caso para distinguirlos: si escribes un artículo sobre "machine learning" y no quieres repetir esa frase exacta, usas `frequency_penalty`. Si no quieres volver a mencionar el mismo ejemplo dos veces, usas `presence_penalty`.

---
## Conclusiones: Guía de Configuración por Tipo de Tarea

In [10]:
# Tabla resumen de configuraciones recomendadas
guia = [
    {
        "tarea": "Generación de código",
        "temperature": 0.1,
        "top_p": 1.0,
        "presence_penalty": 0.0,
        "frequency_penalty": 0.0,
        "razon": "Máxima precisión, el código correcto es determinista"
    },
    {
        "tarea": "Extracción de datos estructurados",
        "temperature": 0.0,
        "top_p": 1.0,
        "presence_penalty": 0.0,
        "frequency_penalty": 0.0,
        "razon": "Consistencia total, el mismo input debe dar el mismo output"
    },
    {
        "tarea": "Documentación técnica",
        "temperature": 0.4,
        "top_p": 1.0,
        "presence_penalty": 0.1,
        "frequency_penalty": 0.3,
        "razon": "Algo de variedad léxica sin perder precisión técnica"
    },
    {
        "tarea": "Brainstorming / generación de ideas",
        "temperature": 1.0,
        "top_p": 0.9,
        "presence_penalty": 0.6,
        "frequency_penalty": 0.3,
        "razon": "Máxima diversidad de ideas, evitar repetir conceptos"
    },
    {
        "tarea": "Análisis y razonamiento",
        "temperature": 0.3,
        "top_p": 1.0,
        "presence_penalty": 0.0,
        "frequency_penalty": 0.1,
        "razon": "Consistencia lógica con ligera variación de vocabulario"
    },
    {
        "tarea": "Contenido creativo (blogs, marketing)",
        "temperature": 0.8,
        "top_p": 1.0,
        "presence_penalty": 0.3,
        "frequency_penalty": 0.5,
        "razon": "Creatividad con diversidad léxica, evitar repeticiones estilísticas"
    },
]

print("=" * 90)
print(f"{'Tarea':<35} {'temp':>6} {'top_p':>6} {'pp':>5} {'fp':>5}")
print("-" * 90)
for g in guia:
    print(f"{g['tarea']:<35} {g['temperature']:>6} {g['top_p']:>6} {g['presence_penalty']:>5} {g['frequency_penalty']:>5}")
    print(f"  → {g['razon']}")
print("=" * 90)

print("""
Notas finales:
- temp y top_p no se ajustan simultáneamente; si cambias uno, deja el otro en default.
- Default recomendado: temperature=0.7, top_p=1.0, pp=0.0, fp=0.0
- Para producción: empieza conservador (temp baja) y sube si las respuestas son demasiado rígidas.
- Los penalties son seguros de combinar entre sí; no son mutuamente excluyentes como temp y top_p.
""")

Tarea                                 temp  top_p    pp    fp
------------------------------------------------------------------------------------------
Generación de código                   0.1    1.0   0.0   0.0
  → Máxima precisión, el código correcto es determinista
Extracción de datos estructurados      0.0    1.0   0.0   0.0
  → Consistencia total, el mismo input debe dar el mismo output
Documentación técnica                  0.4    1.0   0.1   0.3
  → Algo de variedad léxica sin perder precisión técnica
Brainstorming / generación de ideas    1.0    0.9   0.6   0.3
  → Máxima diversidad de ideas, evitar repetir conceptos
Análisis y razonamiento                0.3    1.0   0.0   0.1
  → Consistencia lógica con ligera variación de vocabulario
Contenido creativo (blogs, marketing)    0.8    1.0   0.3   0.5
  → Creatividad con diversidad léxica, evitar repeticiones estilísticas

Notas finales:
- temp y top_p no se ajustan simultáneamente; si cambias uno, deja el otro en default.
- D